In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [ ]:
df=pd.read_csv('train.csv')
df_test=pd.read_csv('test.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.drop('id',axis=1,inplace=True)
# df_test.drop('id',axis=1,inplace=True)

>DataFrame.select_dtypes(include=None, exclude=None) \
>Return a subset of the DataFrame?s columns based on the column dtypes.

In [ ]:
target='NObeyesdad'
obj_col=[x for x in df.select_dtypes(include=['object']) if x !=target]
num_col=[x for x in df.select_dtypes(exclude=['object']) if x !=target]

In [ ]:
print(df['NObeyesdad'].unique())
for i in obj_col:
    print(i,":",df[i].unique())

In [ ]:
feature_dict={
    'Gender':{
        'Female':0,
        'Male':1
    },
    'family_history_with_overweight':{
        'yes':1,
        'no':0
    },
    'FAVC':{
        'yes':1,
        'no':0
    },
    'CAEC':{
        'no':0,
        'Sometimes':1,
        'Frequently':2,
        'Always':3
    },
    'SMOKE':{
        'no':0, 
        'yes':1
    },
    'SCC':{
        'no':1,
        'yes':0
    },
    'CALC':{
       'no':0,
       'Sometimes':1,
       'Always':2 
    },
    'MTRANS' : {
        'Public_Transportation':2,
        'Automobile':4,
        'Walking':0,
        'Motorbike':3,
        'Bike':1
        },
    'NObeyesdad':{
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6,
}
}

obesity_dict = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6,
}

inverse_obesity_dict = {
    0: 'Insufficient_Weight',
    1: 'Normal_Weight',
    2: 'Overweight_Level_I',
    3: 'Overweight_Level_II',
    4: 'Obesity_Type_I',
    5: 'Obesity_Type_II',
    6: 'Obesity_Type_III' 
}

for feature, encoding_map in feature_dict.items():
    print(feature,encoding_map)
    df[feature] = df[feature].map(encoding_map)

In [ ]:
df.head()

In [ ]:
X = df.drop(['NObeyesdad'], axis=1)
y = df['NObeyesdad']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Define XGBClassifier
xgb_0 = XGBClassifier(
    learning_rate=0.1,
    n_estimators=400,
    max_depth=5,
    min_child_weight=1,
    gamma=0,
    colsample_bytree=0.8,
    subsample=0.8,
    objective='multi:softprob',
    nthread=-1,
    random_state=42
)

# Train the classifier
xgb_0.fit(X_train, y_train)

# Make predictions on the test set
predictions = xgb_0.predict(X_test)

# Convert the scikit-learn model to a booster
bst = xgb_0.get_booster()

# Evaluate accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy}")

In [ ]:
df_test.head()

In [ ]:

for feature, encoding_map in feature_dict.items(): 
    if feature !='NObeyesdad':
        df_test[feature] = df_test[feature].map(encoding_map)

In [ ]:
df_test.head()

In [ ]:
prediction_test=xgb_0.predict(df_test.drop('id',axis=1))

In [ ]:
print(len(prediction_test))

In [ ]:
df_test['NObeyesdad']=prediction_test

In [ ]:
df_test.head()

In [ ]:
df_test['NObeyesdad']=df_test['NObeyesdad'].map(inverse_obesity_dict)

In [ ]:
df_test.head()

In [ ]:
submission=df_test[['id','NObeyesdad']]

In [ ]:
submission.to_csv('submission.m1.csv', index = False)